# Capstone build --- Chapter 14: Multi-Agent

The complaint agent is a single governed workflow, and for the capstone that is enough. Chapter~14 is about what changes when one workflow is not: when a task must be decomposed and routed to specialists that report back. The structure is a supervisor that delegates to named workers over a message bus. The complaint harness built in the previous chapters becomes one such worker, unchanged --- the multi-agent layer wraps governed agents, it does not replace their governance.

## A worker wraps the governed harness

A `Worker` pairs a name and a capability with a `GovernanceHarness`. Wrapping the complaint harness in a worker exposes it as a delegable unit: the worker handles a message by running its harness on the delegated task, so the gates, the audit chain and the escalation paths all still apply.

In [ ]:
import json
from pathlib import Path
from agentlab.capstone import build_complaint_harness
from agentlab.multiagent.worker import Worker
from agentlab.multiagent.supervisor import Supervisor

root = next((c for c in (Path('.'), Path('..'), Path('../code'), Path('code'))
             if (c / 'data' / 'eval_cases' / 'cases.json').exists()), Path('.'))
cases = json.loads((root / 'data' / 'eval_cases' / 'cases.json').read_text())
harness, registry = build_complaint_harness(policies_dir=root / 'data' / 'policies')

complaint_worker = Worker(
    name='complaint_handler',
    capability='handle a banking complaint under policy and regulation',
    harness=harness,
)
print('worker     :', complaint_worker.name)
print('capability :', complaint_worker.capability)

## A supervisor delegates over a bus

A `Supervisor` holds the workers and a `MessageBus`. `delegate` sends a typed delegation message to a named worker, which runs its governed harness and returns a response message. The bus records the exchange, so a multi-agent run is as inspectable as a single trajectory.

In [ ]:
supervisor = Supervisor(name='triage', workers=[complaint_worker])
print('workers:', supervisor.workers())

case = next(c for c in cases if c['id'] == 'case-002')
response = supervisor.delegate(
    worker_name='complaint_handler',
    goal='handle complaint',
    inputs={'message': case['message']},
    max_steps=16,
)
print('response from :', response.sender)
print('message type  :', response.message_type)
print('payload keys  :', list(response.payload))

## The exchange is on the bus

The delegation and its response are both messages on the supervisor's bus, so the routing is auditable in the same way the single agent's steps are. The thread below is the record of who asked what of whom.

In [ ]:
for msg in supervisor.bus.all():
    print(f'{msg.sender:16s} -> {msg.receiver:16s} [{msg.message_type}]')

Multi-agent structure is a way to compose governed agents, not a way around their governance: the complaint worker runs the same harness, gates and audit chain whether it is called directly or delegated to. The capstone uses the single agent because its task is one fixed workflow; the supervisor here shows the seam at which it would become a specialist among several. Chapter~15 assembles the single-agent capstone in full.